# Training the bistable-neuron network on MNIST

A two-layer network (784 Poisson-encoded input pixels -> hidden layer ->
output layer, both layers built from the same bistable neuron as notebook
1) trained with surrogate-gradient backprop to classify MNIST digits.

Flow of this notebook:

1. **Parameters** (all in one cell, right below)
2. Build the network and dataloaders
3. **Before training** -- look at the network's activity and weights, untrained
4. **Train**
5. **After training** -- same diagnostics, so you can compare
6. **Training curves** -- accuracy and loss vs. training step

See `bifurcation-analysis` notes (or notebook 1) for *why* this neuron model
can hold a decision across the silent `T_wait` window -- that's what makes
this network different from a standard spiking or rate network.

## Parameters

Everything that controls this run lives here. `params.py` holds the
project-wide defaults (`N_hidden`, `T`, `T_wait`, `lr`, ...); the two
training-length knobs below are overridden for a **short demo run** so this
notebook finishes end-to-end in a couple of minutes. Set them back to
`params.TOTAL_STEPS` / `params.EVAL_EVERY` (or just delete the override) for
a full training run -- expect it to take a good deal longer.

In [ ]:
from params import *   # N_in, N_hidden, N_out, T, T_wait, T_eval, batch_size, lr, ...

# -- demo-run override -------------------------------------------------------
# Full run (params.py default) is TOTAL_STEPS=10000, EVAL_EVERY=50.
# Demo run below is much shorter, just to see the whole pipeline work end to
# end quickly -- accuracy after only ~600 steps will be modest, not final.
DEMO_TOTAL_STEPS = 600
DEMO_EVAL_EVERY  = 50

W0_VALUE = w0   # the neuron's adaptation baseline -- try changing this and re-running

print(f"N_in={N_in}  N_hidden={N_hidden}  N_out={N_out}")
print(f"T={T}  T_wait={T_wait}  T_eval={T_eval}  batch_size={batch_size}  lr={lr}")
print(f"w0={W0_VALUE}")
print(f"demo run: TOTAL_STEPS={DEMO_TOTAL_STEPS}  EVAL_EVERY={DEMO_EVAL_EVERY}")

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import neuron_model
import network_model as netmod
import training_helpers as train_utils
import plotting_helpers as plot_utils

neuron_model.w0 = W0_VALUE   # applies to every bistable_neuron() call that doesn't pass w0 explicitly

print("device:", netmod.device)

## Build the network and data

In [ ]:
train_loader, test_loader = netmod.get_dataloaders(batch_size=batch_size)

netmod.set_seed()   # reproducible weight initialisation
net = netmod.Network()

## Before training -- network activity on a sample digit, untrained weights

One sample from a test batch, run through the untrained network: the input
raster (784 Poisson-encoded pixels), the hidden-layer raster, and the
output-layer raster (10 neurons, one per digit class). With random weights,
the hidden and output layers should look roughly like noise -- no
class-selective structure yet.

In [ ]:
before_activity = plot_utils.get_sample_activity(
    net, test_loader, T=T, poisson_encode_fn=netmod.poisson_encode, device=netmod.device,
)
plot_utils.plot_input_image(before_activity['image'], title=f"label = {before_activity['label']}")
plot_utils.plot_network_rasters(before_activity, title_prefix="before training -- ")
plt.show()

plot_utils.plot_weight_heatmaps(net, title_prefix="before training -- ")
plt.show()

## Train

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr, betas=(0.9, 0.999))
loss_fn = nn.CrossEntropyLoss()

history = train_utils.train_fixed_steps(
    net, train_loader, test_loader, optimizer, loss_fn,
    total_steps=DEMO_TOTAL_STEPS, eval_every=DEMO_EVAL_EVERY, T=T,
)

## After training -- same diagnostics

Same plots as before, now with trained weights. The hidden/output rasters
should show more structure -- in particular, the output layer should settle
onto (and hold, through the `T_wait` silent tail) whichever neuron matches
the true digit label shown above the input image.

In [ ]:
after_activity = plot_utils.get_sample_activity(
    net, test_loader, T=T, poisson_encode_fn=netmod.poisson_encode, device=netmod.device,
)
plot_utils.plot_input_image(after_activity['image'], title=f"label = {after_activity['label']}")
plot_utils.plot_network_rasters(after_activity, title_prefix="after training -- ")
plt.show()

plot_utils.plot_weight_heatmaps(net, title_prefix="after training -- ")
plt.show()

## Training curves -- accuracy and loss

Train (current batch, cheap) vs. test (full test set, every `eval_every`
steps). On the short demo run these will still be climbing, not flat --
that's expected; extend `DEMO_TOTAL_STEPS` (or use the `params.py` full-run
values) to see them converge.

In [ ]:
plot_utils.plot_training_curves(history)

## Next steps

- Re-run from the "Parameters" cell with a different `W0_VALUE` (or a full
  `TOTAL_STEPS` run) and compare the before/after rasters and training
  curves.
- The persistent-memory notebook (`01_neuron_model_overview.ipynb`) is the
  place to build intuition for *why* a given `w0` (and pulse/current
  regime) does or doesn't hold state through `T_wait` -- worth checking
  before spending a long training run on a `w0` that can't hold memory at
  all.